# Zadatak - Vjezba 8: Neuralne mreze - MLP

**Student:** Imad Buljic  
**Predmet:** Rudarenje podataka  
**Tema:** Klasifikacija pomocu MLP modela

U ovom notebooku radim klasifikaciju nad `onlinefoods.csv` datasetom. Dataset je food domen i sadrzi podatke o korisnicima online narucivanja hrane. Cilj je predvidjeti kolonu `Output`, odnosno da li je korisnik narucio hranu. Pratim tok rada iz vjezbe: EDA, preprocessing, `StandardScaler`, poredjenje vise MLP arhitektura, F1 score, classification report, confusion matrix i krive ucenja.


## 1. Uvoz biblioteka

Ucitavaju se biblioteke za obradu podataka, vizualizaciju, enkodiranje kategorickih kolona, skaliranje, treniranje MLP modela i evaluaciju rezultata.


In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.style.use("seaborn-v0_8-whitegrid")

print("Biblioteke su ucitane.")


## 2. Ucitavanje dataseta

Dataset se ucitava iz lokalnog CSV fajla `onlinefoods.csv`, tako da notebook ne zavisi od interneta. Dataset ima numericke kolone kao sto su `Age`, `Family size`, `latitude`, `longitude` i `Pin code`, te vise kategorickih kolona koje ce se kasnije pretvoriti u numericki oblik.


In [ ]:
df_raw = pd.read_csv("onlinefoods.csv")

print(f"Broj redova: {df_raw.shape[0]}")
print(f"Broj kolona: {df_raw.shape[1]}")
print(f"Nedostajuce vrijednosti: {int(df_raw.isna().sum().sum())}")
print("Kolone:")
print(list(df_raw.columns))

display(df_raw.head(10))


## 3. Kratka EDA analiza

Prikazujem raspodjelu target klase i osnovne statistike numerickih kolona. Target je `Output`, gdje `Yes` znaci da je korisnik narucio hranu, a `No` da nije.


In [ ]:
target_counts = df_raw["Output"].value_counts()
display(pd.DataFrame({"Output": target_counts.index, "broj_uzoraka": target_counts.values}))

plt.figure(figsize=(6, 4))
plt.bar(target_counts.index, target_counts.values, color=["#3f7f93", "#d37a46"], edgecolor="black")
plt.title("Raspodjela target klase Output")
plt.xlabel("Output")
plt.ylabel("Broj uzoraka")
plt.tight_layout()
plt.show()

numeric_cols = df_raw.select_dtypes(include=["int64", "float64"]).columns.tolist()
print("Numericke kolone:", numeric_cols)
display(df_raw[numeric_cols].describe().T.round(3))


## 4. Priprema podataka

Kolona `Feedback` se ne koristi kao feature jer predstavlja naknadnu informaciju i mogla bi previse direktno uticati na target. Kolona `Unnamed: 12` je tehnicki visak. Kategoricke kolone pretvaram u numericke pomocu `get_dummies`, nakon cega svi feature-i postaju brojevi.


In [ ]:
df = df_raw.drop(columns=["Feedback", "Unnamed: 12"])

X_raw = df.drop(columns="Output")
y = df["Output"].map({"No": 0, "Yes": 1})

X = pd.get_dummies(X_raw, drop_first=True)

print(f"Oblik prije enkodiranja: {X_raw.shape}")
print(f"Oblik poslije enkodiranja: {X.shape}")
print(f"Target klase: {sorted(y.unique().tolist())} (0=No, 1=Yes)")

display(X.head())


## 5. Train/test split i StandardScaler

Podaci se dijele na trening i test skup uz stratifikaciju, jer target nije potpuno balansiran. `StandardScaler` se fituje samo na trening podacima, zatim se isti scaler koristi za test podatke.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}, y_test:  {y_test.shape}")
print(f"Prosjek nakon skaliranja (prvih 5 feature-a): {np.round(X_train_scaled.mean(axis=0)[:5], 3)}")
print(f"Std nakon skaliranja (prvih 5 feature-a):     {np.round(X_train_scaled.std(axis=0)[:5], 3)}")


## 6. Bonus: MLP bez skaliranja

Za bonus pitanje treniram jednu MLP mrezu nad podacima bez `StandardScaler`. Ovo pokazuje koliko skaliranje moze biti bitno za neuralne mreze.


In [ ]:
mlp_no_scaler = MLPClassifier(
    hidden_layer_sizes=(32,),
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=25,
    random_state=RANDOM_STATE,
)

mlp_no_scaler.fit(X_train, y_train)
pred_no_scaler = mlp_no_scaler.predict(X_test)
f1_no_scaler = f1_score(y_test, pred_no_scaler, average="weighted")

print(f"F1 score bez StandardScaler-a: {f1_no_scaler:.4f}")
print(f"Broj epoha bez skaliranja: {mlp_no_scaler.n_iter_}")


## 7. Treniranje i poredjenje MLP arhitektura

Treniram tri arhitekture nad skaliranim podacima. Minimalni zahtjev zadatka je poredjenje dvije arhitekture, a ovdje dodajem i trecu da se jasnije vidi razlika.


In [ ]:
architectures = {
    "Mala mreza (16)": (16,),
    "Srednja mreza (32)": (32,),
    "Veca mreza (64, 32)": (64, 32),
}

trained_models = {}
results = []

for name, hidden_layers in architectures.items():
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        max_iter=500,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=25,
        random_state=RANDOM_STATE,
    )
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    f1 = f1_score(y_test, y_pred, average="weighted")

    trained_models[name] = {"model": model, "predictions": y_pred}
    results.append({
        "model": name,
        "hidden_layer_sizes": hidden_layers,
        "F1_weighted": round(f1, 4),
        "epohe": model.n_iter_,
        "final_loss": round(model.loss_curve_[-1], 4),
    })

results_df = pd.DataFrame(results).sort_values("F1_weighted", ascending=False).reset_index(drop=True)
display(results_df)

best_name = results_df.loc[0, "model"]
best_model = trained_models[best_name]["model"]
best_pred = trained_models[best_name]["predictions"]

print(f"Najbolji model prema weighted F1: {best_name}")


## 8. Classification report za najbolji model

Za najbolji model prikazujem precision, recall i F1 po klasama. Klasa `No` znaci da korisnik nije narucio, a `Yes` da jeste.


In [ ]:
report_labels = [0, 1]
report_names = ["No", "Yes"]

print(classification_report(
    y_test,
    best_pred,
    labels=report_labels,
    target_names=report_names,
    zero_division=0,
))


## 9. Confusion matrix

Matrica konfuzije pokazuje koliko puta je model pogodio ili promasio svaku klasu.


In [ ]:
cm = confusion_matrix(y_test, best_pred, labels=report_labels)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_title(f"Confusion matrix - {best_name}")
ax.set_xlabel("Predikcija")
ax.set_ylabel("Stvarna klasa")
ax.set_xticks(range(len(report_labels)))
ax.set_yticks(range(len(report_labels)))
ax.set_xticklabels(report_names)
ax.set_yticklabels(report_names)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() * 0.55 else "black")

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


## 10. Krive ucenja

Krive ucenja prikazuju kako se loss mijenja kroz epohe. Ako loss opada, mreza uci; ako stagnira, early stopping zaustavlja treniranje.


In [ ]:
plt.figure(figsize=(10, 5))

for name, values in trained_models.items():
    model = values["model"]
    plt.plot(model.loss_curve_, linewidth=2, label=f"{name} (epohe: {model.n_iter_})")

plt.title("Krive ucenja za MLP arhitekture")
plt.xlabel("Epoha")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()


## 11. Komentar i analiza

**StandardScaler:** Nakon enkodiranja sve kolone jesu numericke, ali nisu na istoj skali. Zbog toga se koristi `StandardScaler`. MLP bez skaliranja je prikazan kao bonus poredjenje i obicno radi losije ili nestabilnije.

**Da li je veca mreza uvijek bolja?** Ne mora biti. Veca mreza ima vise parametara, ali to ne znaci da ce uvijek dati bolji F1 score. Na manjim datasetima moze se desiti da jednostavnija mreza bude stabilnija.

**Koliko epoha je trebalo?** Broj epoha se vidi u tabeli rezultata kroz `n_iter_`. Posto je ukljucen `early_stopping=True`, treniranje se zaustavlja kada validacioni rezultat prestane da se popravlja.

**Je li dataset dobar za zadatak?** Jeste. Dataset nije Iris, ima vise numerickih feature-a nakon enkodiranja i ima klasifikacijski target `Output`. Zbog toga je pogodan za vjezbu sa MLP modelom, skaliranjem, poredjenjem arhitektura, F1 score-om i krivama ucenja.
